In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5"
##################################################################

import logging
import torch
import transformers

from rich.console import Console
from rich.theme import Theme

from src.utils import logging_utils
from src.utils import env_utils

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-03-24 13:47:35 __main__ INFO     torch.__version__='2.6.0+cu124', torch.version.cuda='12.4'
2026-03-24 13:47:35 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=6, torch.cuda.get_device_name()='NVIDIA A100-SXM4-80GB'
2026-03-24 13:47:35 __main__ INFO     transformers.__version__='5.3.0'


## Model

In [3]:
from src.models import ModelandTokenizer

mt = ModelandTokenizer(
    model_key="Qwen/Qwen2.5-Coder-32B-Instruct",
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
rich_print(mt)

2026-03-24 13:47:41 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-03-24 13:47:41 httpx DEBUG    load_ssl_context verify=True cert=None trust_env=True http2=False
2026-03-24 13:47:41 httpx DEBUG    load_verify_locations cafile='/disk/u/arnab/miniconda3/envs/goals/lib/python3.11/site-packages/certifi/cacert.pem'
2026-03-24 13:47:41 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-24 13:47:41 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f7c93cbd0d0>
2026-03-24 13:47:41 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f7c93dd7410> server_hostname='huggingface.co' timeout=10
2026-03-24 13:47:41 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._b

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-03-24 13:48:02 httpcore.connection DEBUG    close.started
2026-03-24 13:48:02 httpcore.connection DEBUG    close.complete
2026-03-24 13:48:02 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-03-24 13:48:02 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f7c8a143f10>
2026-03-24 13:48:02 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f7c93dd7410> server_hostname='huggingface.co' timeout=10
2026-03-24 13:48:02 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f7c986e1290>
2026-03-24 13:48:02 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-03-24 13:48:02 httpcore.http11 DEBUG    send_request_headers.complete
2026-03-24 13:48:02 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-63): 64 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (up_proj): Linear(in_features=5120, out_features=27648, bias=False)
          (down_proj): Linear(in_features=27648, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbedding()
  )
  (lm_head): Linear(in_features=5120, out_features=152064, bias=False)
  (generator): Generator(
    (streamer): Streamer()
  )
)

## Dataset

In [4]:
from src.goals.head_loc_data import HeadLocDataset

ds = HeadLocDataset.load(model_name=mt.name.split("/")[-1])
rich_print(f"Loaded [bold]{len(ds)}[/bold] predicate samples, [bold]{len(ds.pairs)}[/bold] pairs")
SAMPLE_IDX = 0
sample = ds.samples[SAMPLE_IDX]
rich_print(
    f"  predicate_name : {sample.predicate_name}\n",
    f"  variable_type  : {sample.variable_type}\n",
    f"  variable_name  : {sample.variable_name}\n",
    f"  docstring      : {sample.docstring!r}\n",
    f"  python_incomplete_code:\n{sample.python_incomplete_code}\n",
    f"  cpp_incomplete_code:\n{sample.cpp_incomplete_code}\n",
    f"  unit_tests     : {sample.unit_tests[:3]}..."
)

2026-03-24 13:48:09 src.goals.head_loc_data INFO     Loaded 21 predicates, 110 pairs from /disk/u/arnab/Codes/Projects/goals/data/head_loc/pairs/Qwen2.5-Coder-32B-Instruct.json


Loaded 21 predicate samples, 110 pairs

predicate_name : all_even
   variable_type  : list
   variable_name  : lst
   docstring      : 'Return True if every element in lst is even.'
   python_incomplete_code:
def all_even(lst: list) -> bool:
    return
   cpp_incomplete_code:
bool allEven(vector<int> lst) { return
   unit_tests     : [([2, 4, 6], True), ([0, 2, 4], True), ([10, 20, 30], True)]...

## Global Pairs — Oracle-Verified Semantic Compatibility

One-time, model-agnostic step. For each unordered same-type pair (A, B), the oracle
decides if they are semantically distinct enough for contrastive probing.
Results saved to `global_pairs.json`. Run once; re-run only when new predicates are added.

After generating, call `ds.initialize_model_pairs(model_name)` to bootstrap
`pairs/{model_name}.json` for a new model.

In [5]:
# ── Generate global_pairs.json (run once / when predicates change) ────────────
from src.goals.head_loc_oracle import generate_global_pairs

# ds must be loaded first (cell above) — only samples are used, pairs are ignored
global_pairs = generate_global_pairs(ds, oracle_temperature=0.0, save=True)

rich_print(
    f"[bold]Global pairs:[/bold] {len(global_pairs)//2} verified unordered pairs"
    f" → {len(global_pairs)} directed pairs saved to global_pairs.json"
)

from rich.table import Table
table = Table(title="Oracle-Verified Global Pairs", highlight=False)
table.add_column("Source", style="cyan")
table.add_column("Destination", style="magenta")
table.add_column("Type", style="yellow")
seen = set()
for p in global_pairs:
    key = tuple(sorted([p.source_predicate, p.destination_predicate]))
    if key not in seen:
        seen.add(key)
        table.add_row(p.source_predicate, p.destination_predicate, p.variable_type)
rich_print(table)

2026-03-18 17:20:49 src.goals.head_loc_oracle INFO     Type 'list[int]': 7 predicates → 21 oracle calls
2026-03-18 17:20:49 src.goals.head_loc_oracle INFO       [1/63] (all_even, all_positive)
2026-03-18 17:20:49 anthropic._base_client DEBUG    Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-6bb0fdcf-92f6-45b2-bfdd-248d9a05c354', 'content': None, 'json_data': {'max_tokens': 128, 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'You are given two Python predicate functions that take the same input type.\nDetermine whether they are semantically distinct enough to be useful contrastive\npairs in a model probing experiment.\n\nA GOOD pair (answer YES) satisfies all of:\n  - They check for clearly different properties of their input.\n  - A programmer writing one would make clearly different token-level choices\n    than writing the other (e.

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


2026-03-18 17:22:21 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 18 Mar 2026 21:22:21 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'anthropic-ratelimit-input-tokens-limit', b'2000000'), (b'anthropic-ratelimit-input-tokens-remaining', b'2000000'), (b'anthropic-ratelimit-input-tokens-reset', b'2026-03-18T21:22:19Z'), (b'anthropic-ratelimit-output-tokens-limit', b'400000'), (b'anthropic-ratelimit-output-tokens-remaining', b'400000'), (b'anthropic-ratelimit-output-tokens-reset', b'2026-03-18T21:22:21Z'), (b'anthropic-ratelimit-requests-limit', b'4000'), (b'anthropic-ratelimit-requests-remaining', b'3999'), (b'anthropic-ratelimit-requests-reset', b'2026-03-18T21:22:18Z'), (b'anthropic-ratelimit-tokens-limit', b'2400000'), (b'anthropic-ratelimit-tokens-remaining', b'2400000'), (b'anthropic-ratelimit-tokens-reset', b'2026-03-18T21:22:19Z'), (b'request-

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


2026-03-18 17:23:23 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 18 Mar 2026 21:23:23 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'anthropic-ratelimit-input-tokens-limit', b'2000000'), (b'anthropic-ratelimit-input-tokens-remaining', b'2000000'), (b'anthropic-ratelimit-input-tokens-reset', b'2026-03-18T21:23:21Z'), (b'anthropic-ratelimit-output-tokens-limit', b'400000'), (b'anthropic-ratelimit-output-tokens-remaining', b'400000'), (b'anthropic-ratelimit-output-tokens-reset', b'2026-03-18T21:23:23Z'), (b'anthropic-ratelimit-requests-limit', b'4000'), (b'anthropic-ratelimit-requests-remaining', b'3999'), (b'anthropic-ratelimit-requests-reset', b'2026-03-18T21:23:20Z'), (b'anthropic-ratelimit-tokens-limit', b'2400000'), (b'anthropic-ratelimit-tokens-remaining', b'2400000'), (b'anthropic-ratelimit-tokens-reset', b'2026-03-18T21:23:21Z'), (b'request-

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


... (14 outputs left out)

2026-03-18 17:29:39 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 18 Mar 2026 21:29:39 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'anthropic-ratelimit-input-tokens-limit', b'2000000'), (b'anthropic-ratelimit-input-tokens-remaining', b'2000000'), (b'anthropic-ratelimit-input-tokens-reset', b'2026-03-18T21:29:38Z'), (b'anthropic-ratelimit-output-tokens-limit', b'400000'), (b'anthropic-ratelimit-output-tokens-remaining', b'400000'), (b'anthropic-ratelimit-output-tokens-reset', b'2026-03-18T21:29:39Z'), (b'anthropic-ratelimit-requests-limit', b'4000'), (b'anthropic-ratelimit-requests-remaining', b'3999'), (b'anthropic-ratelimit-requests-reset', b'2026-03-18T21:29:37Z'), (b'anthropic-ratelimit-tokens-limit', b'2400000'), (b'anthropic-ratelimit-tokens-remaining', b'2400000'), (b'anthropic-ratelimit-tokens-reset', b'2026-03-18T21:29:38Z'), (b'request-

Global pairs: 55 verified unordered pairs → 110 directed pairs saved to global_pairs.json

2026-03-18 17:29:51 httpcore.connection DEBUG    close.started
2026-03-18 17:29:51 httpcore.connection DEBUG    close.complete
2026-03-18 17:29:51 httpcore.connection DEBUG    close.started
2026-03-18 17:29:51 httpcore.connection DEBUG    close.complete


              Oracle-Verified Global Pairs               
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Source                ┃ Destination            ┃ Type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ all_even              │ all_same               │ list │
│ all_even              │ has_duplicates         │ list │
│ all_even              │ is_sorted              │ list │
│ all_even              │ is_strictly_decreasing │ list │
│ all_even              │ is_strictly_sorted     │ list │
│ all_positive          │ all_same               │ list │
│ all_positive          │ has_duplicates         │ list │
│ all_positive          │ is_sorted              │ list │
│ all_positive          │ is_strictly_decreasing │ list │
│ all_positive          │ is_strictly_sorted     │ list │
│ all_same              │ is_sorted              │ list │
│ all_same              │ is_strictly_decreasing │ list │
│ all_same              │ is_strictly_sorted     │ list │
│ has_duplicates        │ is_sorted              │ list │
│ has_duplicates        │ is_strictly_decreasing │ list │
│ has_duplicates        │ is_strictly_sorted     │ list │
│ ends_with_vowel       │ has_spaces             │ str  │
│ ends_with_vowel       │ is_digit_string        │ str  │
│ ends_with_vowel       │ is_lower               │ str  │
│ ends_with_vowel       │ is_palindrome          │ str  │
│ ends_with_vowel       │ is_upper               │ str  │
│ has_spaces            │ is_digit_string        │ str  │
│ has_spaces            │ is_lower               │ str  │
│ has_spaces            │ is_palindrome          │ str  │
│ has_spaces            │ is_upper               │ str  │
│ has_spaces            │ starts_with_vowel      │ str  │
│ is_digit_string       │ is_lower               │ str  │
│ is_digit_string       │ is_palindrome          │ str  │
│ is_digit_string       │ is_upper               │ str  │
│ is_digit_string       │ starts_with_vowel      │ str  │
│ is_lower              │ is_palindrome          │ str  │
│ is_lower              │ starts_with_vowel      │ str  │
│ is_palindrome         │ is_upper               │ str  │
│ is_palindrome         │ starts_with_vowel      │ str  │
│ is_upper              │ starts_with_vowel      │ str  │
│ is_divisible_by_three │ is_even                │ int  │
│ is_divisible_by_three │ is_odd                 │ int  │
│ is_divisible_by_three │ is_perfect_square      │ int  │
│ is_divisible_by_three │ is_positive            │ int  │
│ is_divisible_by_three │ is_power_of_two        │ int  │
│ is_divisible_by_three │ is_prime               │ int  │
│ is_even               │ is_perfect_square      │ int  │
│ is_even               │ is_positive            │ int  │
│ is_even               │ is_power_of_two        │ int  │
│ is_even               │ is_prime               │ int  │
│ is_odd                │ is_perfect_square      │ int  │
│ is_odd                │ is_positive            │ int  │
│ is_odd                │ is_power_of_two        │ int  │
│ is_odd                │ is_prime               │ int  │
│ is_perfect_square     │ is_positive            │ int  │
│ is_perfect_square     │ is_power_of_two        │ int  │
│ is_perfect_square     │ is_prime               │ int  │
│ is_positive           │ is_power_of_two        │ int  │
│ is_positive           │ is_prime               │ int  │
│ is_power_of_two       │ is_prime               │ int  │
└───────────────────────┴────────────────────────┴──────┘

In [7]:
# ── Bootstrap pairs/{model_name}.json from global_pairs.json ─────────────────
# Run this once per new model, before any pivot mining.
model_name = mt.name.split("/")[-1]
ds.initialize_model_pairs(model_name)
rich_print(f"[green]Initialized pairs/{model_name}.json[/green]")

# Reload ds so ds.pairs reflects the new file
ds = HeadLocDataset.load(model_name=model_name)
rich_print(f"Reloaded: {len(ds.pairs)} pairs")

2026-03-18 17:31:03 src.goals.head_loc_data INFO     Loaded 110 oracle-verified global pairs from /disk/u/arnab/Codes/Projects/goals/data/head_loc/global_pairs.json
2026-03-18 17:31:03 src.goals.head_loc_data INFO     Saved 110 pairs to /disk/u/arnab/Codes/Projects/goals/data/head_loc/pairs/Qwen2.5-Coder-32B-Instruct.json
2026-03-18 17:31:03 src.goals.head_loc_data INFO     Initialized pairs/Qwen2.5-Coder-32B-Instruct.json with 110 global pairs


Initialized pairs/Qwen2.5-Coder-32B-Instruct.json

2026-03-18 17:31:03 src.goals.head_loc_data INFO     Loaded 21 predicates, 110 pairs from /disk/u/arnab/Codes/Projects/goals/data/head_loc/pairs/Qwen2.5-Coder-32B-Instruct.json


Reloaded: 110 pairs

In [ ]:
from rich.table import Table

table = Table(title="Predicate Pairs", highlight=False)
table.add_column("Source (C++)", style="cyan")
table.add_column("Destination (Python)", style="green")
table.add_column("Type", style="yellow")
table.add_column("Oracle verified", justify="center")

for pair in ds.pairs:
    table.add_row(
        pair.source_predicate,
        pair.destination_predicate,
        pair.variable_type,
        "[green]YES[/green]" if pair.oracle_verified else "[dim]no[/dim]",
    )

console.print(table)

                              Predicate Pairs                               
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Source (C++)           ┃ Destination (Python)   ┃ Type ┃ Oracle verified ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ all_even               │ all_same               │ list │       YES       │
│ all_same               │ all_even               │ list │       YES       │
│ all_even               │ has_duplicates         │ list │       YES       │
│ has_duplicates         │ all_even               │ list │       YES       │
│ all_even               │ is_sorted              │ list │       YES       │
│ is_sorted              │ all_even               │ list │       YES       │
│ all_even               │ is_strictly_decreasing │ list │       YES       │
│ is_strictly_decreasing │ all_even               │ list │       YES       │
│ all_even               │ is_strictly_sorted     │ list │       YES       │
│ is_strictly_sorted     │ all_even               │ list │       YES       │
│ all_positive           │ all_same               │ list │       YES       │
│ all_same               │ all_positive           │ list │       YES       │
│ all_positive           │ has_duplicates         │ list │       YES       │
│ has_duplicates         │ all_positive           │ list │       YES       │
│ all_positive           │ is_sorted              │ list │       YES       │
│ is_sorted              │ all_positive           │ list │       YES       │
│ all_positive           │ is_strictly_decreasing │ list │       YES       │
│ is_strictly_decreasing │ all_positive           │ list │       YES       │
│ all_positive           │ is_strictly_sorted     │ list │       YES       │
│ is_strictly_sorted     │ all_positive           │ list │       YES       │
│ all_same               │ is_sorted              │ list │       YES       │
│ is_sorted              │ all_same               │ list │       YES       │
│ all_same               │ is_strictly_decreasing │ list │       YES       │
│ is_strictly_decreasing │ all_same               │ list │       YES       │
│ all_same               │ is_strictly_sorted     │ list │       YES       │
│ is_strictly_sorted     │ all_same               │ list │       YES       │
│ has_duplicates         │ is_sorted              │ list │       YES       │
│ is_sorted              │ has_duplicates         │ list │       YES       │
│ has_duplicates         │ is_strictly_decreasing │ list │       YES       │
│ is_strictly_decreasing │ has_duplicates         │ list │       YES       │
│ has_duplicates         │ is_strictly_sorted     │ list │       YES       │
│ is_strictly_sorted     │ has_duplicates         │ list │       YES       │
│ ends_with_vowel        │ has_spaces             │ str  │       YES       │
│ has_spaces             │ ends_with_vowel        │ str  │       YES       │
│ ends_with_vowel        │ is_digit_string        │ str  │       YES       │
│ is_digit_string        │ ends_with_vowel        │ str  │       YES       │
│ ends_with_vowel        │ is_lower               │ str  │       YES       │
│ is_lower               │ ends_with_vowel        │ str  │       YES       │
│ ends_with_vowel        │ is_palindrome          │ str  │       YES       │
│ is_palindrome          │ ends_with_vowel        │ str  │       YES       │
│ ends_with_vowel        │ is_upper               │ str  │       YES       │
│ is_upper               │ ends_with_vowel        │ str  │       YES       │
│ has_spaces             │ is_digit_string        │ str  │       YES       │
│ is_digit_string        │ has_spaces             │ str  │       YES       │
│ has_spaces             │ is_lower               │ str  │       YES       │
│ is_lower               │ has_spaces             │ str  │       YES       │
│ has_spaces             │ is_palindrome          │ str  │       YES       │
│ is_palindrome          │ has_spaces             │ str  │       YES     

: 

## Stale Codes

### Case 1 — Bare Return Pivot Instances

For each pair, compute the top predicted token at `return ` (bare return, no prefix) for
both dest and target.  Keep pairs where the top tokens differ.

Set `include_docstring=True` to insert each sample's `docstring` field between the
def line and `return`, giving the model a semantic hint.

In [6]:
from src.goals.head_loc_pivot import case1_pivot_instances

# include_docstring=False: no docstring hint (default, cleaner signal)
case1_instances = case1_pivot_instances(mt, ds, include_docstring=False)
rich_print(f"[bold]Case 1:[/bold] {len(case1_instances)} diverging pivot instances")
for pair, pi in case1_instances:
    rich_print(
        f"  {pair.destination_predicate} -> {pair.source_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
        f" | diverges=[green]YES[/green]" if pi.tokens_diverge else " | diverges=[red]no[/red]"
    )


2026-03-18 11:06:31 src.goals.head_loc_pivot DEBUG    Case 1 | is_sorted top=' all' | has_duplicates top=' len' | diverge=True
2026-03-18 11:06:31 src.goals.head_loc_pivot DEBUG    Case 1 | has_duplicates top=' len' | is_sorted top=' all' | diverge=True
2026-03-18 11:06:32 src.goals.head_loc_pivot DEBUG    Case 1 | is_power_of_two top='0' | is_odd top=' n' | diverge=True
2026-03-18 11:06:32 src.goals.head_loc_pivot DEBUG    Case 1 | is_odd top=' n' | is_power_of_two top='0' | diverge=True
2026-03-18 11:06:32 src.goals.head_loc_pivot DEBUG    Case 1 | is_upper top='0' | is_palindrome top=' s' | diverge=True
2026-03-18 11:06:32 src.goals.head_loc_pivot DEBUG    Case 1 | is_palindrome top=' s' | is_upper top='0' | diverge=True
2026-03-18 11:06:32 src.goals.head_loc_pivot INFO     Case 1: 6 diverging pivot instances out of 6 pairs


Case 1: 6 diverging pivot instances

is_sorted -> has_duplicates | prefix='' | dest_top=' all'  targ_top=' len' | diverges=YES

has_duplicates -> is_sorted | prefix='' | dest_top=' len'  targ_top=' all' | diverges=YES

is_power_of_two -> is_odd | prefix='' | dest_top='0'  targ_top=' n' | diverges=YES

is_odd -> is_power_of_two | prefix='' | dest_top=' n'  targ_top='0' | diverges=YES

is_upper -> is_palindrome | prefix='' | dest_top='0'  targ_top=' s' | diverges=YES

is_palindrome -> is_upper | prefix='' | dest_top=' s'  targ_top='0' | diverges=YES

### Case 2 — MCTS-Based Shared Prefix Pivot Instances

MCTS exhaustively explores the generation tree, providing diverse passing
completions for reliable LCP detection. The pipeline is decoupled:

1. **Run MCTS once** (slow, ~200 iters/predicate) — saves results to disk.
2. **Find pivots** (fast, online per model) — reads cached green_completions.

Set  (default) to load cached results from .

In [7]:
from rich.table import Table
from src.goals.head_loc_pivot import run_mcts_for_dataset

mcts_results = run_mcts_for_dataset(
    mt, 
    ds,
    include_docstring = True, 
    overwrite=False, 
    n_iterations=1024, 
    max_depth=32,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

table = Table(title="MCTS Results per Predicate", highlight=False)
table.add_column("Predicate", style="cyan")
table.add_column("Green", justify="right", style="green")
table.add_column("Total", justify="right")
table.add_column("Unique completions", justify="right")
for name, res in mcts_results.items():
    counts = res.eval_counts
    table.add_row(
        name,
        str(counts.get("green", 0)),
        str(counts.get("total", 0)),
        str(len(res.green_completions)),
    )
console.print(table)

2026-03-17 14:31:37 src.goals.head_loc_pivot INFO     run_mcts_for_dataset: save_dir=/disk/u/arnab/Codes/Projects/goals/results/localization/mcts/Qwen2.5-Coder-32B-Instruct
2026-03-17 14:31:37 src.goals.head_loc_pivot INFO       has_duplicates: running MCTS (1024 iters)…
2026-03-17 14:31:42 src.goals.sampling DEBUG    MCTS iteration 50/1024, total nodes: 104
2026-03-17 14:31:46 src.goals.sampling DEBUG    MCTS iteration 100/1024, total nodes: 195
2026-03-17 14:31:48 src.goals.sampling DEBUG    MCTS iteration 150/1024, total nodes: 243
2026-03-17 14:31:48 src.goals.sampling DEBUG    MCTS iteration 200/1024, total nodes: 253
2026-03-17 14:31:50 src.goals.sampling DEBUG    MCTS iteration 250/1024, total nodes: 297
2026-03-17 14:31:51 src.goals.sampling DEBUG    MCTS iteration 300/1024, total nodes: 309
2026-03-17 14:31:52 src.goals.sampling DEBUG    MCTS iteration 350/1024, total nodes: 333
2026-03-17 14:31:54 src.goals.sampling DEBUG    MCTS iteration 400/1024, total nodes: 364
2026-03-1

               MCTS Results per Predicate               
┏━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Predicate       ┃ Green ┃ Total ┃ Unique completions ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ has_duplicates  │    12 │   237 │                  6 │
│ is_odd          │    19 │    67 │                  9 │
│ is_palindrome   │     6 │    84 │                  3 │
│ is_power_of_two │     8 │    93 │                  4 │
│ is_sorted       │     9 │    21 │                  5 │
│ is_upper        │     6 │   109 │                  3 │
└─────────────────┴───────┴───────┴────────────────────┘

#### Tree Inspection (single predicate)

Run MCTS on one sample, evaluate the tree, walk green leaves,
and display the colored HTML tree inline.

In [ ]:
import os
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree
from src.goals.head_loc_pivot import build_python_prompt, _extract_effective_completions

INSPECT_IDX = 2  # change to inspect a different predicate
inspect_sample = ds.samples[INSPECT_IDX]

prompt = build_python_prompt(mt, inspect_sample, pivot_prefix="")

rich_print(f"'[white]{prompt}[/white]'")

In [ ]:
search = MCTSSearch(
    mt,
    prompt,
    max_depth=32,
    top_k=5,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)
search.run(n_iterations=1024, depth_first=True)

In [ ]:
prefix = inspect_sample.python_incomplete_code + " "

eval_counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=inspect_sample.unit_tests,
    function_name=inspect_sample.function_name or inspect_sample.predicate_name,
)
rich_print(f"[bold]{inspect_sample.predicate_name}[/bold] eval_counts: {eval_counts}")

In [ ]:
# Walk green leaves and show effective completions
green_completions = _extract_effective_completions(search.root)
rich_print(f"Unique green completions ({len(green_completions)}):")
for c in green_completions:
    rich_print(f"  [green]{c!r}[/green]")

In [ ]:
# Display the colored MCTS tree inline
import tempfile
from IPython.display import HTML, display

with tempfile.NamedTemporaryFile(suffix=".html", delete=False) as _f:
    _tmp = _f.name
search.save_tree(_tmp, show_colors=True)
with open(_tmp) as _f:
    _html = _f.read()
os.unlink(_tmp)
display(HTML(_html))

In [ ]:
# mcts_results["is_palindrome"].__dict__.keys()
mcts_results["is_palindrome"].green_completions

### Contrastive pairs

In [24]:
from src.goals.head_loc_pivot import build_python_prompt

pair = ds.pairs[-2]
dest_sample = ds.get_sample(pair.destination_predicate)
targ_sample = ds.get_sample(pair.source_predicate)

dest_prompt = build_python_prompt(mt, dest_sample, pivot_prefix="")
targ_prompt = build_python_prompt(mt, targ_sample, pivot_prefix="")

rich_print(f'Destination prompt:\n[white]"{dest_prompt}"[/white]')
rich_print(f'Source prompt:\n[white]"{targ_prompt}"[/white]')

Destination prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Complete the Python function below.<|im_end|>
<|im_start|>assistant
<code>
def is_upper(s: str) -> bool:
    return "

Source prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Complete the Python function below.<|im_end|>
<|im_start|>assistant
<code>
def is_palindrome(s: str) -> bool:
    return "

In [25]:
dest_result = mcts_results.get(pair.destination_predicate)
targ_result = mcts_results.get(pair.source_predicate)

dest_passing = dest_result.green_completions
targ_passing = targ_result.green_completions


rich_print(f"Destination passing completions ({len(dest_passing)}): {dest_passing}")
rich_print(f"Target passing completions ({len(targ_passing)}): {targ_passing}")

NameError: name 'mcts_results' is not defined

In [ ]:
from src.goals.head_loc_pivot import _find_stable_lcps

stable_lcps = _find_stable_lcps(mt, dest_passing, targ_passing, min_stable_k=2)
stable_lcps

In [ ]:
from src.goals.head_loc_pivot import case2_pivot_instances

case2_instances = case2_pivot_instances(
    mt, ds, mcts_results, min_stable_k=1, include_docstring=True
)
rich_print(f"[bold]Case 2:[/bold] {len(case2_instances)} diverging pivot instances")
for pair, pi in case2_instances:
    rich_print(
        f"  {pair.destination_predicate} -> {pair.source_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
    )


In [ ]:
# ── Combine + save is now handled by get_oracle_pivot_instances(save=True) ───
# Use the unified pipeline cell below (get_oracle_pivot_instances) instead.
# It deduplicates across Case 3 + 3b + augmentation and saves into pairs.json.
#
# To manually inspect what's saved:
# ds2 = HeadLocDataset.load(model_name=mt.name.split("/")[-1])
# for pair in ds2.pairs:
#     print(f"{pair.destination_predicate}: {len(pair.pivot_instances)} pivots")


### Augmentation

Dest and target are augmented **jointly** so their pivot prefix tokens stay identical.
`augment_pair` applies the same transformation to both samples with a single call.

In [ ]:
from src.goals.head_loc_augment import augment_pair

dest_name = ds.samples[0].predicate_name
target_name = ds.samples[1].predicate_name if len(ds.samples) > 1 else ds.samples[0].predicate_name

dest = ds[dest_name]
target = ds[target_name]

rich_print(f"[bold]Demo pair:[/bold] dest={dest_name!r} ({dest.variable_name!r}), target={target_name!r} ({target.variable_name!r})")

new_var = "word"
d2, t2 = augment_pair(dest, target, "variable_rename", {"new_var": new_var})

rich_print(f"\n[bold]After variable_rename → {new_var!r}:[/bold]")
rich_print(f"  dest before : {dest.python_incomplete_code!r}")
rich_print(f"  dest after  : {d2.python_incomplete_code!r}")
rich_print(f"  targ before : {target.python_incomplete_code!r}")
rich_print(f"  targ after  : {t2.python_incomplete_code!r}")
rich_print(f"  d2.variable_name={d2.variable_name!r}, t2.variable_name={t2.variable_name!r}")
rich_print(f"  augmentation record: {d2.augmentation}")

In [ ]:
from src.goals.head_loc_augment import generate_pair_variants

variants = generate_pair_variants(dest, target)
rich_print(f"[bold]generate_pair_variants:[/bold] {len(variants)} (dest, target) variant pairs")
for i, (dv, tv) in enumerate(variants):
    aug_type = dv.augmentation[-1]["type"] if dv.augmentation else "unknown"
    rich_print(
        f"  [{i}] {aug_type}"
        f" | dest.variable_name={dv.variable_name!r}"
        f" | targ.variable_name={tv.variable_name!r}"
    )

## Case 3 — Oracle-Based Pivot Instances

**Motivation:** MCTS finds LCPs across all passing implementations of the *destination*
predicate.  When the destination has structurally diverse implementations (e.g. `s.isupper()`
and `s == s.upper()`), the LCP collapses to just `s` — missing the deeper divergence point.

**Fix:** Ask an oracle LLM for canonical implementations of each predicate independently,
then cross-compute LCPs between (dest_impl, target_impl) pairs.  This finds longer,
semantically meaningful shared prefixes like `s == s.` where `.upper()` vs `[::-1]`
first diverge.

**Two gates filter oracle pivots:**
1. **Plausibility gate** — LM assigns ≥ `min_plausibility` to the first LCP token at the bare-return position (the model would naturally reach this prefix).
2. **Divergence gate** — `dest_top_token != target_top_token` at `return {lcp}`.


In [5]:

# ── Inspect oracle pair prompt for the noisy pair ────────────────────────────
from src.goals.head_loc_oracle import (
    build_oracle_pair_prompt,
    get_oracle_lcps,
    get_oracle_implementations,  # kept for standalone inspection
)

INSPECT_PAIR_DEST   = "is_upper"
INSPECT_PAIR_TARGET = "is_palindrome"

# INSPECT_PAIR_DEST = "is_odd"
# INSPECT_PAIR_TARGET = "is_power_of_two"

# INSPECT_PAIR_DEST = "has_duplicates"
# INSPECT_PAIR_TARGET = "is_sorted"

dest_sample_ex = ds.get_sample(INSPECT_PAIR_DEST)
targ_sample_ex = ds.get_sample(INSPECT_PAIR_TARGET)

print("Oracle pair prompt:")
print(f"'''{build_oracle_pair_prompt(dest_sample_ex, targ_sample_ex, n=7, include_docstring=True)}'''")


Oracle pair prompt:
'''You are given two Python predicate functions that both end at `return`.

Find up to 7 shared prefixes P (what immediately follows `return `) such that:
  1. Both predicates can be validly completed starting from that prefix.
  2. The very next token after P is different between the two predicates.
  3. That diverging token directly encodes the functional difference between
     the two predicates — it must be the specific operator, method name, or
     index that makes A and B semantically distinct.

     GOOD divergences: an operator like `<` vs `>`, an index like `0` vs `-1`,
     a method name like `.upper` vs `.lower`.  These tokens, on their own, tell
     you which predicate is being implemented.
     BAD divergences: a variable name like `c` vs `i` (naming convention, not
     semantics), or a wrapper like `not not` that adds no new information.

Reason freely. Each time you confirm a valid prefix, emit it immediately in this
XML tag before continuing to t

In [7]:
from src.goals.head_loc_pivot import build_python_prompt

print(build_python_prompt(mt, sample=dest_sample_ex, pivot_prefix="", include_docstring=True))

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Complete the Python function below.<|im_end|>
<|im_start|>assistant
<code>
def is_upper(s: str) -> bool:
    """Return True if all alphabetic characters in s are uppercase."""
    return 


In [8]:

# ── Oracle-proposed LCPs for the inspected pair ──────────────────────────────
oracle_lcps_ex = get_oracle_lcps(dest_sample_ex, targ_sample_ex, include_docstring=True, n=5, temperature=0.5)

def trim_quotes(s: str) -> str:
    s = s.strip()
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        return s[1:-1]
    return s

oracle_lcps_ex = [trim_quotes(lcp) for lcp in oracle_lcps_ex]

rich_print(
    f"[bold]Oracle LCPs for ({INSPECT_PAIR_DEST}, {INSPECT_PAIR_TARGET}):[/bold]",
    f"[white]{oracle_lcps_ex}[/white]"
) 

2026-03-24 13:49:08 httpx DEBUG    load_ssl_context verify=True cert=None trust_env=True http2=False
2026-03-24 13:49:08 httpx DEBUG    load_verify_locations cafile='/disk/u/arnab/miniconda3/envs/goals/lib/python3.11/site-packages/certifi/cacert.pem'


2026-03-24 13:49:08 anthropic._base_client DEBUG    Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-14f0e97d-5e3a-4c30-ab6f-fee3860f5845', 'content': None, 'json_data': {'max_tokens': 4096, 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'You are given two Python predicate functions that both end at `return`.\n\nFind up to 5 shared prefixes P (what immediately follows `return `) such that:\n  1. Both predicates can be validly completed starting from that prefix.\n  2. The very next token after P is different between the two predicates.\n  3. That diverging token directly encodes the functional difference between\n     the two predicates — it must be the specific operator, method name, or\n     index that makes A and B semantically distinct.\n\n     GOOD divergences: an operator like `<` vs `>`, an index like `0` vs `-1`,\n     a method 

Oracle LCPs for (is_upper, is_palindrome): ['s == s', 'not (s != s', 'bool(s']

In [9]:

# ── Completion gate smoke-test for each oracle LCP ───────────────────────────
# Manually verify: does the model produce valid code from each oracle prefix?
from src.goals.head_loc_oracle import _completion_gate

nl_ids = set(mt.tokenizer.encode("\n", add_special_tokens=False))
stop_ids = {mt.tokenizer.eos_token_id} | nl_ids

for lcp in oracle_lcps_ex:
    ok = _completion_gate(mt, dest_sample_ex, targ_sample_ex, lcp, max_new_tokens=32, include_docstring=True)
    marker = "[green]✓ PASS[/green]" if ok else "[red]✗ FAIL[/red]"
    rich_print(f"  {marker}  lcp={lcp!r}")


2026-03-24 13:49:42 httpcore.connection DEBUG    close.started
2026-03-24 13:49:42 httpcore.connection DEBUG    close.complete
2026-03-24 13:49:47 src.goals.head_loc_oracle DEBUG      eval [is_upper] lcp='s == s'  generated='.upper()'  verdict=PASSED
2026-03-24 13:49:52 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='s == s'  generated='[::-1]'  verdict=PASSED


✓ PASS  lcp='s == s'

2026-03-24 13:49:58 src.goals.head_loc_oracle DEBUG      eval [is_upper] lcp='not (s != s'  generated='.upper() and any(c.isalpha() for c in s))'  verdict=PASSED
2026-03-24 13:50:03 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='not (s != s'  generated='[::-1])'  verdict=PASSED


✓ PASS  lcp='not (s != s'

2026-03-24 13:50:08 src.goals.head_loc_oracle DEBUG      eval [is_upper] lcp='bool(s'  generated=') and all(c.isupper() for c in s if c.isalpha())'  verdict=PASSED
2026-03-24 13:50:14 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='bool(s'  generated=' == s[::-1])'  verdict=PASSED


✓ PASS  lcp='bool(s'

In [10]:
ds.pairs

[PredicatePair(source_predicate='all_even', destination_predicate='all_same', variable_type='list[int]', oracle_verified=True, kl_at_bare_return=0.0, top_token_diverges_at_bare_return=False, pivot_instances=[PivotInstance(pivot_prefix='all(x', dest_top_token=' ==', target_top_token=' %'), PivotInstance(pivot_prefix='not any(x', dest_top_token=' !=', target_top_token=' %'), PivotInstance(pivot_prefix='all(lst[i]', dest_top_token=' ==', target_top_token=' %'), PivotInstance(pivot_prefix='not lst or all(x', dest_top_token=' ==', target_top_token=' %'), PivotInstance(pivot_prefix='lst', dest_top_token=' ==', target_top_token=' and'), PivotInstance(pivot_prefix='not', dest_top_token=' lst', target_top_token=' any'), PivotInstance(pivot_prefix='bool(all(x', dest_top_token=' ==', target_top_token=' %'), PivotInstance(pivot_prefix='not not all(x', dest_top_token=' ==', target_top_token=' %'), PivotInstance(pivot_prefix='True and all(x', dest_top_token=' ==', target_top_token=' %'), PivotInstan

In [11]:
# ── Full Case 3 run across all pairs ─────────────────────────────────────────
from src.goals.head_loc_oracle import case3_pivot_instances

case3_instances = case3_pivot_instances(
    mt,
    ds,
    include_docstring=True,
    n_oracle=5,
    oracle_temperature=0.5,
    max_completion_tokens=32,
    pairs=[ds.pairs[-2]]
)

rich_print(f"[bold]Case 3:[/bold] {len(case3_instances)} diverging pivot instances")
for pair, pi in case3_instances:
    rich_print(
        f"  {pair.destination_predicate} -> {pair.source_predicate}"
        f" | prefix={pi.pivot_prefix!r}"
        f" | dest_top={pi.dest_top_token!r}  targ_top={pi.target_top_token!r}"
    )


2026-03-18 16:47:23 src.goals.head_loc_oracle INFO     Case 3: pair is_upper -> is_palindrome


2026-03-18 16:47:23 src.utils.oracle_llms DEBUG    found cached claude response for 9a15c64358b0aa0b2f887c150c81b07d - loading
2026-03-18 16:47:23 src.goals.head_loc_oracle DEBUG    Oracle pair response for (is_upper, is_palindrome):
I need to find shared prefixes where both functions can be validly completed, but the next token diverges meaningfully.

Let me think about natural implementations:

**Function A** (`is_upper`): `return s.isupper()` or `return all(c.isupper() for c in s if c.isalpha())`

**Function B** (`is_palindrome`): `return s == s[::-1]` or `return s == s[::-1]` or `return all(s[i] == s[-(i+1)] for i in range(len(s)//2))`

---

**Candidate 1:** Prefix `s`

- A: `s.isupper()`
- B: `s == s[::-1]`
- Next token after `s`: A=`.` B=` ==` (or `==`)
- The diverging token `.` vs `==` is meaningful (method call vs equality comparison). Valid.

<pivot>
PREFIX: s
A: s.isupper()
B: s == s[::-1]
DIVERGES: A='.'  B='=='
</pivot>

---

**Candidate 2:** Prefix `all(`

- A: `all(c.isup

Case 3: 3 diverging pivot instances

is_upper -> is_palindrome | prefix='s' | dest_top='.is'  targ_top=' =='

is_upper -> is_palindrome | prefix='s == s' | dest_top='.upper'  targ_top='[::-'

is_upper -> is_palindrome | prefix='all(c' | dest_top='.is'  targ_top=' =='

In [12]:
 # ── Semantic-preserving augmentation of Case 3 pivot instances ──────────────
from src.goals.head_loc_oracle import augment_pivot_instances, AUGMENTATION_WRAPPERS

print(f"Wrappers: {AUGMENTATION_WRAPPERS}")
print(f"Base instances: {len(case3_instances)}")

aug_instances = augment_pivot_instances(
    mt,
    case3_instances,
    ds,
    # wrappers=AUGMENTATION_WRAPPERS,  # default; override to try a subset
    include_docstring=False,
    max_completion_tokens=32,
)

print(f"Augmented instances: {len(aug_instances)}")
print(f"Total Case 3 (base + aug): {len(case3_instances) + len(aug_instances)}")

# Preview
from rich.table import Table
table = Table(title="Augmented Pivot Instances", highlight=False)
table.add_column("Dest", style="cyan")
table.add_column("Target", style="magenta")
table.add_column("Prefix", style="yellow")
table.add_column("dest→", style="green")
table.add_column("targ→", style="red")
for pair, pi in aug_instances:
    table.add_row(
        pair.destination_predicate,
        pair.source_predicate,
        repr(pi.pivot_prefix),
        repr(pi.dest_top_token),
        repr(pi.target_top_token),
    )
rich_print(table)


Wrappers: ['bool(', 'int(', 'not not ', 'True and ', 'False or ']
Base instances: 3
2026-03-18 16:47:51 src.goals.head_loc_oracle DEBUG      eval [is_upper] lcp='bool(s'  generated=') and all(c.isupper() for c in s)'  verdict=PASSED
2026-03-18 16:47:54 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='bool(s'  generated=' == s[::-1])'  verdict=PASSED
2026-03-18 16:47:55 src.goals.head_loc_oracle INFO       ✓ Augmented pivot: is_upper -> is_palindrome prefix='bool(s' | dest=')' targ=' =='
2026-03-18 16:47:58 src.goals.head_loc_oracle DEBUG      eval [is_upper] lcp='int(s'  generated='.isupper())'  verdict=FAILED (failed tests: ['Input: HELLO, Expected: True, Actual: 1, Status: (✗), Error: None', 'Input: hello, Expected: False, Actual: 0, Status: (✗), Error: None', 'Input: Hello, Expected: False, Actual: 0, Status: (✗), Error: None', 'Input: aBC, Expected: False, Actual: 0, Status: (✗), Error: None'])
2026-03-18 16:47:58 src.goals.head_loc_oracle DEBUG      Augmentation comp

                     Augmented Pivot Instances                      
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓
┃ Dest     ┃ Target        ┃ Prefix            ┃ dest→    ┃ targ→  ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩
│ is_upper │ is_palindrome │ 'bool(s'          │ ')'      │ ' =='  │
│ is_upper │ is_palindrome │ 'True and s'      │ '.is'    │ ' =='  │
│ is_upper │ is_palindrome │ 'False or s'      │ '.is'    │ ' =='  │
│ is_upper │ is_palindrome │ 'bool(s == s'     │ '.upper' │ '[::-' │
│ is_upper │ is_palindrome │ 'not not s == s'  │ '.upper' │ '[::-' │
│ is_upper │ is_palindrome │ 'True and s == s' │ '.upper' │ '[::-' │
│ is_upper │ is_palindrome │ 'False or s == s' │ '.upper' │ '[::-' │
│ is_upper │ is_palindrome │ 'bool(all(c'      │ '.is'    │ ' =='  │
│ is_upper │ is_palindrome │ 'not not all(c'   │ '.is'    │ ' =='  │
│ is_upper │ is_palindrome │ 'True and all(c'  │ '.is'    │ ' =='  │
│ is_upper │ is_palindrome │ 'False or all(c'  │ '.is'    │ ' =='  │
└──────────┴───────────────┴───────────────────┴──────────┴────────┘

In [11]:

# # ── Combine Case 1 + 2 + 3, deduplicate, and save ────────────────────────────
# seen = set()
# combined = []
# for pi in case1_instances + case2_instances + case3_instances:
#     key = (pi.destination_predicate, pi.target_predicate, pi.pivot_prefix)
#     if key not in seen:
#         seen.add(key)
#         combined.append(pi)

# assert all(pi.tokens_diverge for pi in combined), "All instances must diverge"

# rich_print(
#     f"Total: Case1={len(case1_instances)} + Case2={len(case2_instances)}"
#     f" + Case3={len(case3_instances)} = {len(combined)} after dedup"
# )

# # Save
# ds.save_pivot_instances(combined)
# rich_print("[green]Saved pivot_instances.json with Case 3 augmentation[/green]")

# loaded = HeadLocDataset.load_pivot_instances()
# rich_print(f"Round-trip: loaded {len(loaded)} instances")


## Case 3b — Per-Predicate Oracle + Mechanical LCP

**Motivation:** Case 3 gives the oracle *both* predicates and asks it to reason about
divergence points directly.  Case 3b is complementary: query the oracle *independently*
for each predicate to get a diverse set of concrete implementations, then compute
token-level LCPs mechanically across the full cross-product of (dest_impl, targ_impl) pairs.

**Pipeline:**
1. `get_oracle_implementations(dest_sample)` → list of dest implementations.
2. `get_oracle_implementations(targ_sample)` → list of target implementations.
3. Cross-product all pairs → compute `_lcp_tokens` for each → keep LCPs ≥ `min_prefix_len` chars.
4. Extract diverging tokens at position `len(tok_lcp)`; reject if either is a single-letter
   variable name (`_is_meaningful_token`).
5. Deduplicate LCPs, then apply the same completion + divergence gates as Case 3.

**Advantage over Case 3:** mechanically verifiable LCPs from real implementations; doesn't
rely on the oracle's ability to reason about cross-predicate divergence.
**Disadvantage:** misses longer shared prefixes that require semantic reasoning
(e.g. `s == s.` where no single implementation uses exactly that prefix).

In [14]:

# ── Inspect per-predicate oracle implementations ──────────────────────────────
from src.goals.head_loc_oracle import get_oracle_implementations, _is_meaningful_token
from src.goals.head_loc_pivot import _lcp_tokens

INSPECT_3B_DEST   = "is_upper"
INSPECT_3B_TARGET = "is_palindrome"

dest_sample_3b = ds.get_sample(INSPECT_3B_DEST)
targ_sample_3b = ds.get_sample(INSPECT_3B_TARGET)

dest_impls_3b = get_oracle_implementations(dest_sample_3b, n=8, temperature=0.5)
targ_impls_3b = get_oracle_implementations(targ_sample_3b, n=8, temperature=0.5)

rich_print(f"[bold]Dest ({INSPECT_3B_DEST}) implementations:[/bold]")
for impl in dest_impls_3b:
    rich_print(f"  [cyan]{impl}[/cyan]")

rich_print(f"[bold]Target ({INSPECT_3B_TARGET}) implementations:[/bold]")
for impl in targ_impls_3b:
    rich_print(f"  [magenta]{impl}[/magenta]")


2026-03-18 16:49:12 src.utils.oracle_llms DEBUG    found cached claude response for cb6fcc5fa25e6f596a21f79a8cb25026 - loading
2026-03-18 16:49:12 src.goals.head_loc_oracle DEBUG    Oracle response for 'is_upper':
- s.isupper()
- s == s.upper()
- all(c.isupper() for c in s if c.isalpha())
- not any(c.islower() for c in s)
- s.upper() == s and any(c.isalpha() for c in s)
- bool(s) and s == s.upper() and s.lower() != s
- all(not c.islower() for c in s) and any(c.isalpha() for c in s)
- s.upper() == s != s.lower()
2026-03-18 16:49:12 src.goals.head_loc_oracle INFO     Oracle implementations for 'is_upper' (8 unique): ['s.isupper()', 's == s.upper()', 'all(c.isupper() for c in s if c.isalpha())', 'not any(c.islower() for c in s)', 's.upper() == s and any(c.isalpha() for c in s)', 'bool(s) and s == s.upper() and s.lower() != s', 'all(not c.islower() for c in s) and any(c.isalpha() for c in s)', 's.upper() == s != s.lower()']
2026-03-18 16:49:12 src.utils.oracle_llms DEBUG    found cached cl

Dest (is_upper) implementations:

s.isupper()

s == s.upper()

all(c.isupper() for c in s if c.isalpha())

not any(c.islower() for c in s)

... (9 outputs left out)

s == s[::-1] if s else True

not any(s != s[~i] for i in range(len(s) // 2))

tuple(s) == tuple(s)[::-1]

s.lower() == s.lower()[::-1]

In [15]:

# ── Inspect cross-product LCPs + diverging token filter ──────────────────────
from rich.table import Table

table = Table(title=f"Cross-product LCPs: {INSPECT_3B_DEST} × {INSPECT_3B_TARGET}", highlight=False)
table.add_column("dest impl", style="cyan")
table.add_column("targ impl", style="magenta")
table.add_column("LCP", style="yellow")
table.add_column("div dest", style="green")
table.add_column("div targ", style="red")
table.add_column("keep?", style="bold")

for impl_dest in dest_impls_3b:
    for impl_targ in targ_impls_3b:
        lcp = _lcp_tokens(mt, impl_dest, impl_targ)
        if len(lcp) < 3:
            continue
        tok_lcp  = mt.tokenizer.encode(lcp, add_special_tokens=False)
        tok_dest = mt.tokenizer.encode(impl_dest, add_special_tokens=False)
        tok_targ = mt.tokenizer.encode(impl_targ, add_special_tokens=False)
        n = len(tok_lcp)
        if n >= len(tok_dest) or n >= len(tok_targ):
            continue
        div_d = mt.tokenizer.decode([tok_dest[n]])
        div_t = mt.tokenizer.decode([tok_targ[n]])
        keep = _is_meaningful_token(div_d) and _is_meaningful_token(div_t)
        table.add_row(impl_dest, impl_targ, repr(lcp), repr(div_d), repr(div_t), "✓" if keep else "✗")

rich_print(table)


                                   Cross-product LCPs: is_upper × is_palindrome                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┓
┃ dest impl                         ┃ targ impl                         ┃ LCP       ┃ div dest ┃ div targ ┃ keep? ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━┩
│ s == s.upper()                    │ s == s[::-1]                      │ 's == s'  │ '.upper' │ '[::-'   │ ✓     │
│ s == s.upper()                    │ s == ''.join(reversed(s))         │ 's =='    │ ' s'     │ " ''."   │ ✗     │
│ s == s.upper()                    │ s == s[::-1] if s else True       │ 's == s'  │ '.upper' │ '[::-'   │ ✓     │
│ all(c.isupper() for c in s if     │ all(s == s[~i] for i in           │ 'all'     │ '(c'     │ '(s'     │ ✓     │
│ c.isalpha())                      │ range(len(s) // 2))               │           │          │          │       │
│ not any(c.islower() for c in s)   │ not any(s != s[~i] for i in       │ 'not any' │ '(c'     │ '(s'     │ ✓     │
│                                   │ range(len(s) // 2))               │           │          │          │       │
│ all(not c.islower() for c in s)   │ all(s == s[~i] for i in           │ 'all'     │ '(not'   │ '(s'     │ ✓     │
│ and any(c.isalpha() for c in s)   │ range(len(s) // 2))               │           │          │          │       │
└───────────────────────────────────┴───────────────────────────────────┴───────────┴──────────┴──────────┴───────┘

In [16]:
# ── Full Case 3b run across all pairs ────────────────────────────────────────
from src.goals.head_loc_oracle import case3b_pivot_instances

case3b_instances = case3b_pivot_instances(
    mt,
    ds,
    include_docstring=False,
    n_implementations=8,
    oracle_temperature=0.5,
    min_prefix_len=3,
    max_completion_tokens=32,
    pairs=[ds.pairs[-2]],   # smoke-test on one pair; remove to run all
)

rich_print(f"[bold]Case 3b:[/bold] {len(case3b_instances)} diverging pivot instances")

table = Table(title="Case 3b Pivot Instances", highlight=False)
table.add_column("Dest", style="cyan")
table.add_column("Target", style="magenta")
table.add_column("Prefix", style="yellow")
table.add_column("dest→", style="green")
table.add_column("targ→", style="red")
for pair, pi in case3b_instances:
    table.add_row(
        pair.destination_predicate,
        pair.source_predicate,
        repr(pi.pivot_prefix),
        repr(pi.dest_top_token),
        repr(pi.target_top_token),
    )
rich_print(table)


2026-03-18 16:49:12 src.goals.head_loc_oracle INFO     Case 3b: pair is_upper -> is_palindrome
2026-03-18 16:49:12 src.utils.oracle_llms DEBUG    found cached claude response for cb6fcc5fa25e6f596a21f79a8cb25026 - loading


2026-03-18 16:49:12 src.goals.head_loc_oracle DEBUG    Oracle response for 'is_upper':
- s.isupper()
- s == s.upper()
- all(c.isupper() for c in s if c.isalpha())
- not any(c.islower() for c in s)
- s.upper() == s and any(c.isalpha() for c in s)
- bool(s) and s == s.upper() and s.lower() != s
- all(not c.islower() for c in s) and any(c.isalpha() for c in s)
- s.upper() == s != s.lower()
2026-03-18 16:49:12 src.goals.head_loc_oracle INFO     Oracle implementations for 'is_upper' (8 unique): ['s.isupper()', 's == s.upper()', 'all(c.isupper() for c in s if c.isalpha())', 'not any(c.islower() for c in s)', 's.upper() == s and any(c.isalpha() for c in s)', 'bool(s) and s == s.upper() and s.lower() != s', 'all(not c.islower() for c in s) and any(c.isalpha() for c in s)', 's.upper() == s != s.lower()']
2026-03-18 16:49:12 src.utils.oracle_llms DEBUG    found cached claude response for a59fe58d5020ea8c4db3e0426dc5f724 - loading
2026-03-18 16:49:12 src.goals.head_loc_oracle DEBUG    Oracle resp

Case 3b: 3 diverging pivot instances

                  Case 3b Pivot Instances                   
┏━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┓
┃ Dest     ┃ Target        ┃ Prefix    ┃ dest→    ┃ targ→  ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━┩
│ is_upper │ is_palindrome │ 's == s'  │ '.upper' │ '[::-' │
│ is_upper │ is_palindrome │ 'all'     │ '(c'     │ '(s'   │
│ is_upper │ is_palindrome │ 'not any' │ '(c'     │ '(s'   │
└──────────┴───────────────┴───────────┴──────────┴────────┘

## Testing `_tokens_differ_meaningfully` (Case 3b naming filter)

`_tokens_differ_meaningfully` normalizes each diverging token by replacing non-keyword/non-builtin identifiers with a `VAR` placeholder, then checks whether the normalized forms differ. If they don't, the divergence is naming-only (e.g. `(s` vs `(c` both normalize to `(VAR`) and the candidate is rejected.

In [12]:
from src.goals.head_loc_oracle import _normalize_token, _tokens_differ_meaningfully

# ------------------------------------------------------------------
# Unit test: _normalize_token
# ------------------------------------------------------------------
normalize_cases = [
    # (token, expected_normalized, description)
    ("(s",    "(VAR",  "open-paren + identifier → VAR"),
    ("(c",    "(VAR",  "open-paren + different identifier → VAR"),
    ("(self", "(VAR",  "open-paren + 'self' (not a keyword/builtin) → VAR"),
    ("<",     "<",     "operator: unchanged"),
    (">",     ">",     "operator: unchanged"),
    ("len(",  "len(",  "'len' is a builtin → preserved"),
    ("if",    "if",    "'if' is a keyword → preserved"),
    ("return","return","'return' is a keyword → preserved"),
    ("timeout","VAR",  "user-defined name → VAR"),
    ("host",  "VAR",   "user-defined name → VAR"),
    ("True",  "True",  "'True' is a keyword → preserved"),
    ("None",  "None",  "'None' is a keyword → preserved"),
]

print("=== _normalize_token ===")
all_ok = True
for token, expected, desc in normalize_cases:
    got = _normalize_token(token)
    ok = got == expected
    if not ok:
        all_ok = False
    status = "✓" if ok else f"✗ expected={expected!r}"
    print(f"  {token!r:15s} → {got!r:15s}  {status}  ({desc})")

print(f"\n{'All normalization tests passed!' if all_ok else 'SOME TESTS FAILED'}")


=== _normalize_token ===
  '(s'            → '(VAR'           ✓  (open-paren + identifier → VAR)
  '(c'            → '(VAR'           ✓  (open-paren + different identifier → VAR)
  '(self'         → '(VAR'           ✓  (open-paren + 'self' (not a keyword/builtin) → VAR)
  '<'             → '<'              ✓  (operator: unchanged)
  '>'             → '>'              ✓  (operator: unchanged)
  'len('          → 'len('           ✓  ('len' is a builtin → preserved)
  'if'            → 'if'             ✓  ('if' is a keyword → preserved)
  'return'        → 'return'         ✓  ('return' is a keyword → preserved)
  'timeout'       → 'VAR'            ✓  (user-defined name → VAR)
  'host'          → 'VAR'            ✓  (user-defined name → VAR)
  'True'          → 'True'           ✓  ('True' is a keyword → preserved)
  'None'          → 'None'           ✓  ('None' is a keyword → preserved)

All normalization tests passed!


In [15]:
# ------------------------------------------------------------------
# Unit test: _tokens_differ_meaningfully
# ------------------------------------------------------------------
differ_cases = [
    # (tok_dest, tok_targ, expected_result, description)
    # --- should be REJECTED (naming-only, returns False) ---
    ("(s",     "(c",      False, "both → '(VAR': naming-only → reject"),
    ("(self",  "(node",   False, "both → '(VAR': naming-only → reject"),
    ("timeout","host",    False, "both → 'VAR': naming-only → reject"),
    ("my_var", "x_4",       False, "both → 'VAR': naming-only → reject"),
    # --- should be KEPT (semantically distinct, returns True) ---
    ("<",      ">",       True,  "operators differ → keep"),
    ("len(",   "(s",      True,  "'len(' vs '(VAR': builtin preserved → keep"),
    ("if",     "(s",      True,  "keyword vs identifier → keep"),
    ("return", "yield",   True,  "two different keywords → keep"),
    ("True",   "False",   True,  "two different keywords → keep"),
    ("not",    "(s",      True,  "keyword vs identifier → keep"),
    ("[",      "(",       True,  "different punctuation → keep"),
    ("=",      "==",      True,  "different operators → keep"),
]

print("=== _tokens_differ_meaningfully ===")
all_ok = True
for tok_dest, tok_targ, expected, desc in differ_cases:
    got = _tokens_differ_meaningfully(tok_dest, tok_targ)
    ok = got == expected
    if not ok:
        all_ok = False
    action = "keep" if got else "reject"
    status = "✓" if ok else f"✗ expected={expected}"
    print(f"  {tok_dest!r:12s} vs {tok_targ!r:12s} → {str(got):5s} ({action:6s})  {status}  {desc}")

print(f"\n{'All differ-meaningfully tests passed!' if all_ok else 'SOME TESTS FAILED'}")


=== _tokens_differ_meaningfully ===
  '(s'         vs '(c'         → False (reject)  ✓  both → '(VAR': naming-only → reject
  '(self'      vs '(node'      → False (reject)  ✓  both → '(VAR': naming-only → reject
  'timeout'    vs 'host'       → False (reject)  ✓  both → 'VAR': naming-only → reject
  'my_var'     vs 'x_4'        → False (reject)  ✓  both → 'VAR': naming-only → reject
  '<'          vs '>'          → True  (keep  )  ✓  operators differ → keep
  'len('       vs '(s'         → True  (keep  )  ✓  'len(' vs '(VAR': builtin preserved → keep
  'if'         vs '(s'         → True  (keep  )  ✓  keyword vs identifier → keep
  'return'     vs 'yield'      → True  (keep  )  ✓  two different keywords → keep
  'True'       vs 'False'      → True  (keep  )  ✓  two different keywords → keep
  'not'        vs '(s'         → True  (keep  )  ✓  keyword vs identifier → keep
  '['          vs '('          → True  (keep  )  ✓  different punctuation → keep
  '='          vs '=='         → Tru

In [17]:
rich_print(pair)

PredicatePair(
    source_predicate='all_even',
    destination_predicate='all_same',
    variable_type='list[int]',
    oracle_verified=True,
    kl_at_bare_return=0.0,
    top_token_diverges_at_bare_return=False,
    pivot_instances=[
        PivotInstance(pivot_prefix='all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='not any(x', dest_top_token=' !=', target_top_token=' %'),
        PivotInstance(pivot_prefix='all(lst[i]', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='not lst or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='lst', dest_top_token=' ==', target_top_token=' and'),
        PivotInstance(pivot_prefix='not', dest_top_token=' lst', target_top_token=' any'),
        PivotInstance(pivot_prefix='bool(all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='not not all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='True and all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='False or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='bool(not any(x', dest_top_token=' !=', target_top_token=' %'),
        PivotInstance(pivot_prefix='True and not any(x', dest_top_token=' !=', target_top_token=' %'),
        PivotInstance(pivot_prefix='False or not any(x', dest_top_token=' !=', target_top_token=' %'),
        PivotInstance(pivot_prefix='bool(all(lst[i]', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='not not all(lst[i]', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='True and all(lst[i]', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='False or all(lst[i]', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='bool(not lst or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='not not not lst or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='True and not lst or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='False or not lst or all(x', dest_top_token=' ==', target_top_token=' %'),
        PivotInstance(pivot_prefix='bool(not', dest_top_token=' lst', target_top_token=' any'),
        PivotInstance(pivot_prefix='False or not', dest_top_token=' lst', target_top_token=' any')
    ]
)

In [19]:
# ------------------------------------------------------------------
# End-to-end demo: inspect Case 3b raw candidates for a specific pair
# and show which ones would be filtered by _tokens_differ_meaningfully.
#
# Uses _mine_case3b_raw (internal helper) directly so we can see the
# pre-gate diverging tokens. No model or oracle call needed.
# ------------------------------------------------------------------
from src.goals.head_loc_oracle import _tokens_differ_meaningfully

# Grab a specific pair from the dataset
pair_name_src = "is_palindrome"
pair_name_dst = "is_lower"

# Find the matching predicate pair in ds
target_pair = None
for pair in ds.pairs:
    src_name = pair.source_predicate
    dst_name = pair.destination_predicate
    if src_name == pair_name_src and dst_name == pair_name_dst:
        target_pair = pair
        break
    if src_name == pair_name_dst and dst_name == pair_name_src:
        target_pair = pair
        break

if target_pair is None:
    print(f"Pair {pair_name_src}/{pair_name_dst} not found in dataset.")
    print("Available pairs (first 10):")
    for p in ds.pairs[:10]:
        print(f"  {p.source_predicate} → {p.destination_predicate}")
else:
    src_fn = target_pair.source_predicate
    dst_fn = target_pair.destination_predicate
    print(f"Found pair: {src_fn} → {dst_fn}")
    print(f"Existing pivot instances: {len(target_pair.pivot_instances)}")
    if target_pair.pivot_instances:
        print("\nExisting instances:")
        for pi in target_pair.pivot_instances[:5]:
            print(f"  pivot_prefix={pi.pivot_prefix!r:50s}  dest_tok={pi.dest_top_token!r}  tgt_tok={pi.target_top_token!r}")
            div_dest = pi.dest_top_token
            div_targ = pi.target_top_token
            differs = _tokens_differ_meaningfully(div_dest, div_targ)
            print(f"    → differs_meaningfully={differs}  (would be {'kept' if differs else 'REJECTED by new gate'})")


Found pair: is_lower → is_palindrome
Existing pivot instances: 19

Existing instances:
  pivot_prefix='s == s'                                            dest_tok='[::-'  tgt_tok='.lower'
    → differs_meaningfully=True  (would be kept)
  pivot_prefix='all'                                               dest_tok='(s'  tgt_tok='(c'
    → differs_meaningfully=False  (would be REJECTED by new gate)
  pivot_prefix='not any'                                           dest_tok='(s'  tgt_tok='(char'
    → differs_meaningfully=False  (would be REJECTED by new gate)
  pivot_prefix='s.lower() == s'                                    dest_tok='[::-'  tgt_tok='\n'
    → differs_meaningfully=True  (would be kept)
  pivot_prefix='bool(s == s'                                       dest_tok='[::-'  tgt_tok='.lower'
    → differs_meaningfully=True  (would be kept)


## `get_oracle_pivot_instances` — unified pipeline

Single call that runs Case 3 + Case 3b mining, Gate Check 1, augmentation (Gate Check 2),
deduplication, and save.  Replace the individual `case3_pivot_instances` + `aug_instances`
cells above with this.

In [23]:
target_pair

PredicatePair(source_predicate='is_lower', destination_predicate='is_palindrome', variable_type='str', oracle_verified=True, kl_at_bare_return=0.0, top_token_diverges_at_bare_return=False, pivot_instances=[PivotInstance(pivot_prefix='s == s', dest_top_token='[::-', target_top_token='.lower'), PivotInstance(pivot_prefix='all', dest_top_token='(s', target_top_token='(c'), PivotInstance(pivot_prefix='not', dest_top_token=' s', target_top_token=' any'), PivotInstance(pivot_prefix='bool(s == s', dest_top_token='[::-', target_top_token='.lower'), PivotInstance(pivot_prefix='not not s == s', dest_top_token='[::-', target_top_token='.lower'), PivotInstance(pivot_prefix='True and s == s', dest_top_token='[::-', target_top_token='.lower'), PivotInstance(pivot_prefix='False or s == s', dest_top_token='[::-', target_top_token='.lower'), PivotInstance(pivot_prefix='bool(all', dest_top_token='(s', target_top_token='(c'), PivotInstance(pivot_prefix='not not all', dest_top_token='(s', target_top_token

In [24]:
# ── Unified oracle pivot pipeline (smoke-test on one pair) ───────────────────
from src.goals.head_loc_oracle import get_oracle_pivot_instances

oracle_instances = get_oracle_pivot_instances(
    mt,
    ds,
    pairs=[target_pair],          # smoke-test; remove to run all pairs
    include_case3=True,
    include_case3b=True,
    include_augmentation=True,
    include_docstring=True,
    oracle_include_docstring=True,
    n_oracle=5,
    oracle_temperature=0.5,
    max_completion_tokens=32,
    n_implementations=8,
    min_prefix_len=3,
    save=False,
)

rich_print(
    f"[bold]get_oracle_pivot_instances:[/bold]"
    f" {len(oracle_instances)} total pivot instances (base + augmented, deduped)"
)

from rich.table import Table

table = Table(title="Oracle Pivot Instances", highlight=False)
table.add_column("Dest", style="cyan")
table.add_column("Target", style="magenta")
table.add_column("Prefix", style="yellow")
table.add_column("dest→", style="green")
table.add_column("targ→", style="red")
for pair, pi in oracle_instances:
    table.add_row(
        pair.destination_predicate,
        pair.source_predicate,
        repr(pi.pivot_prefix),
        repr(pi.dest_top_token),
        repr(pi.target_top_token),
    )
rich_print(table)


2026-03-24 14:04:40 httpx DEBUG    load_ssl_context verify=True cert=None trust_env=True http2=False
2026-03-24 14:04:40 httpx DEBUG    load_verify_locations cafile='/disk/u/arnab/miniconda3/envs/goals/lib/python3.11/site-packages/certifi/cacert.pem'
2026-03-24 14:04:40 src.utils.oracle_llms DEBUG    found cached claude response for bb33c958e64de7395e0dd6602be2817d - loading
2026-03-24 14:04:40 src.goals.head_loc_oracle DEBUG    Oracle pair response for (is_palindrome, is_lower):
I need to find shared prefixes where both functions can be validly completed, but diverge at a semantically meaningful token.

Let me think about natural implementations:

**Function A** (is_palindrome): `return s == s[::-1]`
**Function B** (is_lower): `return s.islower()`

Or alternatively:
- A: `return s == s[::-1]`
- B: `return s == s.lower()`

Or:
- A: `return all(s[i] == s[-(i+1)] for i in range(len(s)//2))`
- B: `return all(c.islower() for c in s if c.isalpha())`

Let me look for shared prefixes:

---

*

2026-03-24 14:04:40 src.goals.head_loc_oracle INFO       4 candidate LCPs for 'is_palindrome' -> 'is_lower'
2026-03-24 14:04:40 src.goals.head_loc_oracle INFO     Stage 1 [is_palindrome->is_lower]: 5 raw → 4 deduped candidates
2026-03-24 14:04:46 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='s == s'  generated='[::-1]'  verdict=PASSED
2026-03-24 14:04:51 src.goals.head_loc_oracle DEBUG      eval [is_lower] lcp='s == s'  generated='.lower()'  verdict=PASSED
2026-03-24 14:04:51 src.goals.head_loc_oracle DEBUG      lcp='s == s' | dest_top='[::-' | targ_top='.lower' | diverge=True
2026-03-24 14:04:51 src.goals.head_loc_oracle INFO       ✓ Validated: is_palindrome->is_lower prefix='s == s' | dest='[::-' targ='.lower'
2026-03-24 14:04:57 src.goals.head_loc_oracle DEBUG      eval [is_palindrome] lcp='s =='  generated=' s[::-1]'  verdict=PASSED
2026-03-24 14:05:02 src.goals.head_loc_oracle DEBUG      eval [is_lower] lcp='s =='  generated=' s.lower()'  verdict=PASSED
2026-03-24

get_oracle_pivot_instances: 6 total pivot instances (base + augmented, deduped)

                       Oracle Pivot Instances                       
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┓
┃ Dest          ┃ Target   ┃ Prefix            ┃ dest→  ┃ targ→    ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━┩
│ is_palindrome │ is_lower │ 's == s'          │ '[::-' │ '.lower' │
│ is_palindrome │ is_lower │ 'not'             │ ' s'   │ ' any'   │
│ is_palindrome │ is_lower │ 'bool(s == s'     │ '[::-' │ '.lower' │
│ is_palindrome │ is_lower │ 'not not s == s'  │ '[::-' │ '.lower' │
│ is_palindrome │ is_lower │ 'True and s == s' │ '[::-' │ '.lower' │
│ is_palindrome │ is_lower │ 'False or s == s' │ '[::-' │ '.lower' │
└───────────────┴──────────┴───────────────────┴────────┴──────────┘